# Task 17: BGE-M3 Query Protocol Benchmark

**Goal**: Correct the BGE-M3 query encoding — remove the `"Query: "` prefix that Task 16 incorrectly used.

**Key insight**: Document embeddings are identical regardless of query prefix. Only the 20 evaluation queries need re-encoding.

**Environment**: Kaggle Notebook with GPU. Reuses Task 16 FAISS index (`task16_artifacts/bge_m3_faiss.index`).

**Comparison**:
- A. Dense with `query_prefix="Query: "` (Task 16 reference)
- B. Dense with `query_prefix=""` (corrected)
- C. BM25+source baseline
- D. Hybrid RRF with corrected dense

## Cell 1: Environment info

In [ ]:
import sys, platform, time, json
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
try:
    import torch
    print(f"torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("torch not installed")

## Cell 2: Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-gpu 2>/dev/null || pip install -q sentence-transformers faiss-cpu

import sentence_transformers, faiss, numpy
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"faiss: {faiss.__version__}")
print(f"numpy: {numpy.__version__}")

## Cell 3: Paths

**Upload to Kaggle**:
1. The `svk-corpus/` folder (same as Task 16)
2. The `task16_artifacts/` folder from Task 16 output (contains `bge_m3_faiss.index`)

Edit `CORPUS_DIR` and `TASK16_ARTIFACTS_DIR` below.

In [ ]:
from pathlib import Path

# --- EDIT THESE PATHS ---
CORPUS_DIR = Path("/kaggle/input/datasets/priyanshu47/jainllm")
TASK16_ARTIFACTS_DIR = Path("/kaggle/input/jainllm-task16-artifacts/task16_artifacts")

# Verify
print("Corpus files:")
for p in [
    CORPUS_DIR / "data" / "release" / "rag_corpus.jsonl",
    CORPUS_DIR / "manifests" / "source_manifest.csv",
    CORPUS_DIR / "configs" / "retrieval_eval_queries_v1.json",
]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p}")

print("Task 16 artifacts:")
for p in [
    TASK16_ARTIFACTS_DIR / "bge_m3_faiss.index",
    TASK16_ARTIFACTS_DIR / "text_ids.json",
]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p}")

## Cell 4: Add svk-corpus/src to Python path

In [ ]:
import sys
src_dir = str(CORPUS_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from svk_corpus.retrieval import (
    BM25Index, SentenceTransformerDense, UnavailableDense,
    load_corpus, RetrievalResult
)
from svk_corpus.retrieval.rerank import Reranker
from svk_corpus.retrieval.source_channel import SourceIndex, SourceUnitExpander, build_source_index
from svk_corpus.retrieval.contract import CanonicalResult, RetrievalChannel, validate_result
print("All imports OK")

## Cell 5: Load corpus and queries

In [ ]:
import json, time

rag_path = CORPUS_DIR / "data" / "release" / "rag_corpus.jsonl"
manifest_path = CORPUS_DIR / "manifests" / "source_manifest.csv"
queries_path = CORPUS_DIR / "configs" / "retrieval_eval_queries_v1.json"

t0 = time.perf_counter()
docs = load_corpus(rag_path, manifest_path)
t_load = time.perf_counter() - t0
print(f"Corpus loaded: {len(docs)} units in {t_load:.1f}s")

queries = json.loads(queries_path.read_text(encoding="utf-8"))["queries"]
print(f"Evaluation queries: {len(queries)}")

## Cell 6: Load BAAI/bge-m3 model

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"

print(f"Loading model: {MODEL_NAME}")
t0 = time.perf_counter()
model = SentenceTransformer(MODEL_NAME)
t_model = time.perf_counter() - t0
print(f"Model loaded in {t_model:.1f}s")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## Cell 7: Load Task 16 FAISS index (no doc re-encoding)

In [ ]:
import faiss, numpy as np

faiss_path = TASK16_ARTIFACTS_DIR / "bge_m3_faiss.index"
text_ids_path = TASK16_ARTIFACTS_DIR / "text_ids.json"

t0 = time.perf_counter()
index = faiss.read_index(str(faiss_path))
t_load_index = time.perf_counter() - t0

print(f"FAISS index loaded in {t_load_index:.2f}s")
print(f"  Vectors: {index.ntotal}")
print(f"  Dimension: {index.d}")

# Load text_id mapping
text_ids = json.loads(text_ids_path.read_text())
assert len(text_ids) == index.ntotal, f"text_ids count {len(text_ids)} != index count {index.ntotal}"
print(f"  Text IDs: {len(text_ids)}")

# Build text_id -> doc index mapping for fast lookup
text_id_to_idx = {tid: i for i, tid in enumerate(text_ids)}
print(f"  text_id -> index mapping built")

## Cell 8: Dense search helper (works with loaded FAISS index)

In [ ]:
def dense_search(query: str, top_k: int = 10, query_prefix: str = "") -> list[tuple[int, float]]:
    """Encode query with given prefix, search FAISS, return (doc_index, score)."""
    q_vec = model.encode(
        [query_prefix + query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    scores, indices = index.search(q_vec, min(top_k, index.ntotal))
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append((int(idx), float(score)))
    return results

print("dense_search() defined")

## Cell 9: Metrics helpers

In [ ]:
TOP_K = 10
RRF_K = 60

def compute_metrics(queries, hit_sources_per_q, top_k=10):
    recalls5, recalls10, rr = [], [], []
    zero = 0
    for q, hits in zip(queries, hit_sources_per_q):
        if q.get("relevance") != "source-level":
            continue
        expected = set(q.get("expected_source_ids") or [])
        if not expected:
            continue
        f5 = len(expected & set(hits[:5])) / len(expected)
        f10 = len(expected & set(hits[:top_k])) / len(expected)
        first = next((i for i, s in enumerate(hits, 1) if s in expected), None)
        recalls5.append(f5)
        recalls10.append(f10)
        rr.append(1.0 / first if first else 0.0)
        if not (expected & set(hits)):
            zero += 1
    return {
        "recall@5": round(sum(recalls5) / len(recalls5), 3) if recalls5 else None,
        "recall@10": round(sum(recalls10) / len(recalls10), 3) if recalls10 else None,
        "mrr": round(sum(rr) / len(rr), 3) if rr else None,
        "queries_with_zero_relevant": zero,
        "n_source_level": len(recalls5),
    }

def per_query_detail(queries, hit_sources_per_q, top_k=10):
    details = []
    for q, hits in zip(queries, hit_sources_per_q):
        entry = {"id": q["id"], "category": q.get("category", ""),
                 "relevance": q.get("relevance", ""), "query": q["query"][:80]}
        if q.get("relevance") == "source-level":
            expected = set(q.get("expected_source_ids") or [])
            entry["expected"] = sorted(expected)
            entry["found_in_top5"] = sorted(expected & set(hits[:5]))
            entry["found_in_top10"] = sorted(expected & set(hits[:top_k]))
            entry["missed"] = sorted(expected - set(hits[:top_k]))
            entry["top_sources"] = hits[:top_k]
        else:
            entry["top_sources"] = hits[:5]
        details.append(entry)
    return details

print("Metrics helpers defined")

## Cell 10: Dense-only with prefix="Query: " (Task 16 reference)

In [ ]:
print("Evaluating dense-only (prefix='Query: ', Task 16 reference)...")
ref_hits = []
ref_latencies = []
for q in queries:
    t0 = time.perf_counter()
    hits = dense_search(q["query"], top_k=TOP_K, query_prefix="Query: ")
    dt = time.perf_counter() - t0
    ref_latencies.append(dt)
    ref_hits.append([docs[idx].source_id for idx, _ in hits])

ref_metrics = compute_metrics(queries, ref_hits, TOP_K)
ref_metrics["mean_latency_s"] = round(sum(ref_latencies) / len(ref_latencies), 4)
print(f"  Dense (prefix='Query: '): R@5={ref_metrics['recall@5']}  R@10={ref_metrics['recall@10']}  "
      f"MRR={ref_metrics['mrr']}  zero={ref_metrics['queries_with_zero_relevant']}  "
      f"lat={ref_metrics['mean_latency_s']}s")

## Cell 11: Dense-only with prefix="" (corrected)

In [ ]:
print("Evaluating dense-only (prefix='', corrected)...")
corr_hits = []
corr_latencies = []
for q in queries:
    t0 = time.perf_counter()
    hits = dense_search(q["query"], top_k=TOP_K, query_prefix="")
    dt = time.perf_counter() - t0
    corr_latencies.append(dt)
    corr_hits.append([docs[idx].source_id for idx, _ in hits])

corr_metrics = compute_metrics(queries, corr_hits, TOP_K)
corr_metrics["mean_latency_s"] = round(sum(corr_latencies) / len(corr_latencies), 4)
print(f"  Dense (prefix=''):          R@5={corr_metrics['recall@5']}  R@10={corr_metrics['recall@10']}  "
      f"MRR={corr_metrics['mrr']}  zero={corr_metrics['queries_with_zero_relevant']}  "
      f"lat={corr_metrics['mean_latency_s']}s")

## Cell 12: BM25+source baseline (regression check)

In [ ]:
print("Evaluating BM25+source baseline (C)...")
t0 = time.perf_counter()
bm25 = BM25Index(docs)
t_bm25 = time.perf_counter() - t0
print(f"  BM25 index built in {t_bm25:.1f}s")

sidx = build_source_index(manifest_path)
expander = SourceUnitExpander(docs)
diversify = Reranker(strategy="diversify", max_per_source=3)

bm25_hit_sources = []
bm25_latencies = []
for q in queries:
    t0 = time.perf_counter()
    pool = bm25.search(q["query"], top_k=2000)
    unit_tier = diversify.rerank([RetrievalResult(doc=h.doc, score=h.score,
                                                   rank=h.rank, methods={"bm25": h.rank})
                                   for h in pool])[:TOP_K]
    smatches = sidx.search(q["query"], top_k=TOP_K)
    missing = [m for m in smatches
               if m.source_id not in {r.doc.source_id for r in unit_tier}
               and not m.identifier_only]
    n_fill = min(3, len(missing))
    fill = expander.expand(missing[:n_fill], top_k=n_fill)
    keep = max(TOP_K - len(fill), 0)
    combined = (list(unit_tier[:keep]) + fill)[:TOP_K]
    dt = time.perf_counter() - t0
    bm25_latencies.append(dt)
    bm25_hit_sources.append([r.doc.source_id for r in combined])

bm25_metrics = compute_metrics(queries, bm25_hit_sources, TOP_K)
bm25_metrics["mean_latency_s"] = round(sum(bm25_latencies) / len(bm25_latencies), 4)
print(f"  BM25+src C: R@5={bm25_metrics['recall@5']}  R@10={bm25_metrics['recall@10']}  "
      f"MRR={bm25_metrics['mrr']}  zero={bm25_metrics['queries_with_zero_relevant']}  "
      f"lat={bm25_metrics['mean_latency_s']}s")

## Cell 13: Hybrid RRF with corrected dense

In [ ]:
print("Evaluating hybrid RRF (BM25 + corrected dense)...")
hybrid_hits = []
hybrid_latencies = []
for q in queries:
    t0 = time.perf_counter()
    # BM25 results
    bm25_pool = bm25.search(q["query"], top_k=TOP_K * 3)
    bm25_ranks = {r.doc.text_id: i + 1 for i, r in enumerate(bm25_pool)}
    # Dense results (corrected: no prefix)
    dense_hits = dense_search(q["query"], top_k=TOP_K * 3, query_prefix="")
    dense_ranks = {}
    for rank, (idx, _) in enumerate(dense_hits, 1):
        dense_ranks[docs[idx].text_id] = rank
    # RRF fusion
    all_ids = set(bm25_ranks.keys()) | set(dense_ranks.keys())
    fused = []
    for tid in all_ids:
        r1 = bm25_ranks.get(tid, TOP_K * 3 + 1)
        r2 = dense_ranks.get(tid, TOP_K * 3 + 1)
        rrf_score = 1.0 / (RRF_K + r1) + 1.0 / (RRF_K + r2)
        doc = None
        for r in bm25_pool:
            if r.doc.text_id == tid:
                doc = r.doc
                break
        if doc is None:
            for idx, _ in dense_hits:
                if docs[idx].text_id == tid:
                    doc = docs[idx]
                    break
        if doc is not None:
            fused.append((rrf_score, doc))
    fused.sort(key=lambda t: (-t[0], t[1].text_id))
    dt = time.perf_counter() - t0
    hybrid_latencies.append(dt)
    hybrid_hits.append([doc.source_id for _, doc in fused[:TOP_K]])

hybrid_metrics = compute_metrics(queries, hybrid_hits, TOP_K)
hybrid_metrics["mean_latency_s"] = round(sum(hybrid_latencies) / len(hybrid_latencies), 4)
print(f"  Hybrid RRF: R@5={hybrid_metrics['recall@5']}  R@10={hybrid_metrics['recall@10']}  "
      f"MRR={hybrid_metrics['mrr']}  zero={hybrid_metrics['queries_with_zero_relevant']}  "
      f"lat={hybrid_metrics['mean_latency_s']}s")

## Cell 14: Comparison table

In [ ]:
print("=" * 70)
print("TASK 17 RESULTS: BGE-M3 Query Protocol Benchmark")
print("=" * 70)
print(f"{'Mode':<30} {'R@5':>6} {'R@10':>6} {'MRR':>6} {'Zero':>5} {'Lat(s)':>7}")
print("-" * 70)
for name, m in [
    ("Dense (prefix='Query: ')", ref_metrics),
    ("Dense (prefix='')", corr_metrics),
    ("BM25+src baseline", bm25_metrics),
    ("Hybrid (corrected dense)", hybrid_metrics),
]:
    print(f"{name:<30} {m['recall@5']:>6.3f} {m['recall@10']:>6.3f} {m['mrr']:>6.3f} "
          f"{m['queries_with_zero_relevant']:>5} {m['mean_latency_s']:>7.4f}")

print()
print("Delta (corrected vs Task 16 reference):")
if ref_metrics["recall@5"] and corr_metrics["recall@5"]:
    print(f"  R@5:  {corr_metrics['recall@5'] - ref_metrics['recall@5']:+.3f}")
if ref_metrics["recall@10"] and corr_metrics["recall@10"]:
    print(f"  R@10: {corr_metrics['recall@10'] - ref_metrics['recall@10']:+.3f}")
if ref_metrics["mrr"] and corr_metrics["mrr"]:
    print(f"  MRR:  {corr_metrics['mrr'] - ref_metrics['mrr']:+.3f}")

print()
print("Delta (hybrid corrected vs BM25 baseline):")
if bm25_metrics["recall@5"] and hybrid_metrics["recall@5"]:
    print(f"  R@5:  {hybrid_metrics['recall@5'] - bm25_metrics['recall@5']:+.3f}")
if bm25_metrics["recall@10"] and hybrid_metrics["recall@10"]:
    print(f"  R@10: {hybrid_metrics['recall@10'] - bm25_metrics['recall@10']:+.3f}")
if bm25_metrics["mrr"] and hybrid_metrics["mrr"]:
    print(f"  MRR:  {hybrid_metrics['mrr'] - bm25_metrics['mrr']:+.3f}")

## Cell 15: Per-query comparison (reference vs corrected)

In [ ]:
print("Per-query source-level comparison (dense-only):")
print(f"{'Query':<6} {'Ref R@5':>8} {'Corr R@5':>9} {'Delta':>7} {'Ref R@10':>9} {'Corr R@10':>10} {'Delta':>7}")
print("-" * 65)
for q, r_h, c_h in zip(queries, ref_hits, corr_hits):
    if q.get("relevance") != "source-level":
        continue
    expected = set(q.get("expected_source_ids") or [])
    if not expected:
        continue
    r5 = len(expected & set(r_h[:5])) / len(expected)
    c5 = len(expected & set(c_h[:5])) / len(expected)
    r10 = len(expected & set(r_h[:10])) / len(expected)
    c10 = len(expected & set(c_h[:10])) / len(expected)
    d5 = c5 - r5
    d10 = c10 - r10
    marker = " *" if abs(d5) > 0.001 or abs(d10) > 0.001 else ""
    print(f"{q['id']:<6} {r5:>8.3f} {c5:>9.3f} {d5:>+7.3f} {r10:>9.3f} {c10:>10.3f} {d10:>+7.3f}{marker}")

## Cell 16: Highlighted queries (q06, q09, q10, q11, q17)

In [ ]:
highlight_ids = {"q06", "q09", "q10", "q11", "q17"}
print("Highlighted queries (q06, q09, q10, q11, q17):")
print()
for q, r_h, c_h in zip(queries, ref_hits, corr_hits):
    if q["id"] not in highlight_ids:
        continue
    expected = set(q.get("expected_source_ids") or [])
    r5 = len(expected & set(r_h[:5])) / len(expected) if expected else 0
    c5 = len(expected & set(c_h[:5])) / len(expected) if expected else 0
    r10 = len(expected & set(r_h[:10])) / len(expected) if expected else 0
    c10 = len(expected & set(c_h[:10])) / len(expected) if expected else 0
    print(f"{q['id']}: {q['query']}")
    print(f"  Expected: {sorted(expected)}")
    print(f"  prefix='Query: ' -> R@5={r5:.3f} R@10={r10:.3f}")
    print(f"  prefix=''        -> R@5={c5:.3f} R@10={c10:.3f}")
    print(f"  Found (corrected): {sorted(expected & set(c_h[:10]))}")
    print(f"  Missed (corrected): {sorted(expected - set(c_h[:10]))}")
    print()

## Cell 17: Contract validation (corrected dense results)

In [ ]:
# Validate corrected dense results against Task 14 contract
sample_q = queries[0]
sample_hits = dense_search(sample_q["query"], top_k=3, query_prefix="")
print("Contract validation (corrected dense results):")
for idx, score in sample_hits:
    doc = docs[idx]
    cr = CanonicalResult(
        result_id=doc.text_id,
        source_id=doc.source_id,
        text=doc.text,
        title=doc.title,
        retrieval_channel=RetrievalChannel.DENSE,
        retrieval_rank=1,
        retrieval_score=score,
        dense_rank=1,
        dense_score=score,
    )
    violations = validate_result(cr)
    print(f"  {doc.text_id} (score={score:.4f}): {'VALID' if not violations else violations}")

## Cell 18: Save full report

In [ ]:
OUT_DIR = Path("/kaggle/working/task17_artifacts")
OUT_DIR.mkdir(exist_ok=True)

try:
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
except:
    gpu_name = "unknown"

report = {
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "label": "task17_bge_m3_query_protocol",
    "environment": {
        "python": sys.version,
        "torch": torch.__version__,
        "sentence_transformers": sentence_transformers.__version__,
        "faiss": faiss.__version__,
        "numpy": numpy.__version__,
        "gpu": gpu_name,
    },
    "model": {
        "name": MODEL_NAME,
        "embedding_dim": index.d,
        "normalize_embeddings": True,
        "max_seq_length": 1024,
    },
    "task16_reference": {
        "query_prefix": "Query: ",
        "metrics": ref_metrics,
        "per_query": per_query_detail(queries, ref_hits),
    },
    "corrected": {
        "query_prefix": "",
        "metrics": corr_metrics,
        "per_query": per_query_detail(queries, corr_hits),
    },
    "bm25_source_baseline": {
        "metrics": bm25_metrics,
        "per_query": per_query_detail(queries, bm25_hit_sources),
    },
    "hybrid_rrf_corrected": {
        "metrics": hybrid_metrics,
        "per_query": per_query_detail(queries, hybrid_hits),
    },
    "delta_corrected_vs_reference": {
        "recall@5": round(corr_metrics["recall@5"] - ref_metrics["recall@5"], 3)
                    if corr_metrics["recall@5"] and ref_metrics["recall@5"] else None,
        "recall@10": round(corr_metrics["recall@10"] - ref_metrics["recall@10"], 3)
                    if corr_metrics["recall@10"] and ref_metrics["recall@10"] else None,
        "mrr": round(corr_metrics["mrr"] - ref_metrics["mrr"], 3)
               if corr_metrics["mrr"] and ref_metrics["mrr"] else None,
    },
    "delta_hybrid_vs_baseline": {
        "recall@5": round(hybrid_metrics["recall@5"] - bm25_metrics["recall@5"], 3)
                    if hybrid_metrics["recall@5"] and bm25_metrics["recall@5"] else None,
        "recall@10": round(hybrid_metrics["recall@10"] - bm25_metrics["recall@10"], 3)
                    if hybrid_metrics["recall@10"] and bm25_metrics["recall@10"] else None,
        "mrr": round(hybrid_metrics["mrr"] - bm25_metrics["mrr"], 3)
               if hybrid_metrics["mrr"] and bm25_metrics["mrr"] else None,
    },
}

report_path = OUT_DIR / "task17_bge_m3_query_protocol.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Report saved to {report_path}")
print(f"File size: {report_path.stat().st_size / 1e3:.0f} KB")

## Cell 19: Done

In [ ]:
print("Task 17 complete.")
print(f"Artifacts: {OUT_DIR}")
print("To download: Kaggle -> Output -> task17_artifacts/")
print("Copy task17_bge_m3_query_protocol.json to svk-corpus/data/reports/")